# Lab: Multi-Class Classification

*In this lab, we will build a multi-class classification model using scikit-learn, guided by the [Foundational Methodology for Data Science](../../../05_methodology/). While real-world projects require comprehensive documentation at each stage, this lab focuses on a streamlined, practical approach to demonstrate the core concepts and workflow. Therefore we will settle with the summaries of hypothetical stage reports.*

---

> ### 📝 1. Business Understanding Report (Summary)
> * **Business Context:** A botanical research institute has discovered that the *Iris-virginica* species possesses unique genetic properties that are valuable for developing new floral hybrids. The institute needs to collect pure samples of this species for their research.
> * **Business Problem:** The current process of identifying *Iris-virginica* relies on manual inspection, which is slow and costly. While [an initial proposal](../02_linear_and_logistic_regression/10_logistic_regression_lab.ipynb) was to build a simple binary classifier ("Is it Virginica?"), it has been determined that this is insufficient. A misclassification of another species (e.g., *Iris-setosa*) as *Iris-virginica* would contaminate the research samples, wasting significant time and resources. Therefore, the business needs a more precise system that can differentiate between all three species to ensure only pure *Iris-virginica* samples are selected.
> * **Project Goal:** To build a **multi-class classification model** that uses the physical measurements of an iris flower (sepal length, petal width, etc.) to automatically and accurately classify a specimen into one of the three distinct species: *Iris-setosa*, *Iris-versicolor*, or *Iris-virginica*.

---

> ### 📝 2. Analytic Approach Report (Summary)
> *   **Problem Type:** ***Supervised Multi-Class Classification***. The objective is no longer to predict a binary "yes/no" outcome but to assign each flower to one of three distinct and mutually exclusive categories (*Iris-setosa*, *Iris-versicolor*, or *Iris-virginica*).
> *   **Candidate Models:** To comprehensively evaluate the best approach for this new problem, we will implement and compare three distinct strategies:
>     1.  **Baseline (Native Multi-Class):** **Softmax Regression**. This is the natural extension of Logistic Regression for multi-class problems and will serve as our interpretable baseline.
>     2.  **Candidate 2 (Wrapper):** **One-vs-Rest (OvR) Classifier**. This strategy involves training one binary classifier for each class against all others.
>     3.  **Candidate 3 (Wrapper):** **One-vs-One (OvO) Classifier**. This strategy involves training a dedicated binary classifier for every possible pair of classes.
> *   **Evaluation Metrics:** Given the balanced nature of the Iris dataset and the need for high-confidence predictions, we will use a suite of metrics:
>     *   **Accuracy:** To provide a single, overall measure of the models' correctness.
>     *   **Confusion Matrix:** This is critical for visualizing the specific types of errors the model makes (e.g., how many *versicolor* samples are misclassified as *virginica*).
>     *   **Precision, Recall, and F1-Score (per class and weighted average):** To provide a nuanced evaluation of performance for each species and an aggregate score that accounts for class balance.
> *   **Validation Strategy:** The modeling process will involve splitting the data into training and testing sets (80/20 split) and applying **feature scaling** (`StandardScaler`), as the underlying linear models benefit from standardized features. The performance of all three candidate models will be compared on the held-out test set.

---

> ### 📝 3. Data Requirements Report (Summary)
> *   **Data Source:** The project will continue to use the classic `iris.csv` dataset, sourced from the Scikit-learn library (`sklearn.datasets.load_iris`), as established in the [previous lab](../02_linear_and_logistic_regression/10_logistic_regression_lab.ipynb). The dataset contains data on 150 iris flower samples.
> *   **Features (Independent Variables):** The features remain unchanged: `sepal_length`, `sepal_width`, `petal_length`, and `petal_width` (all measurements in centimeters).
> *   **Target (Dependent Variable):** This is the key change from the previous analysis. The target variable is now the original multi-class `species` column (or `target` from the scikit-learn object), which contains three distinct classes: *setosa* (0), *versicolor* (1), and *virginica* (2). We will no longer be engineering a binary `is_virginica` variable.
> *   **Granularity & Privacy:** The data granularity remains at the individual flower specimen level. The dataset is a well-known, public, and anonymized benchmark, containing no sensitive or personally identifiable information (PII), so there are no privacy concerns.

---

## Stage 4: Data Collection
As usual, we start with importing the necessary libraries and configuring them.

In [10]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsOneClassifier, OneVsRestClassifier 
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay

# Configurations
plt.style.use("fivethirtyeight")
sns.set_theme(style="white", palette="colorblind")

### 4.1. Extract: Collect and Store Raw Data
First, we **Extract** the data from its original source (`scikit-learn`) and store it in an untouched, "raw" format in the `/data/raw/` directory. This creates a perfect, versionable mirror of the source system at the time of collection.

> ⚠️ **Caution:** To follow best practices for reproducibility, we are re-extracting the data from scratch rather than using files from the previous lab. This ensures our project is self-contained.

In [11]:
# Extract data from the source system (scikit-learn)
raw_data = load_iris()
print(f"Data has been extracted from the source system as '{type(raw_data)}'")

# Create a DataFrame with the original, untouched column names
raw_df = pd.DataFrame(data=raw_data.data, columns=raw_data.feature_names)
raw_df['target'] = raw_data.target

# Define and create the raw data directory
raw_data_dir = Path("../data/raw")
raw_data_dir.mkdir(parents=True, exist_ok=True)

# Define the file path and save the raw DataFrame
raw_data_path = raw_data_dir / "iris_raw_v1.csv"
raw_df.to_csv(raw_data_path, index=False)

print(f"Raw, untouched data has been saved to: {raw_data_path}")
raw_df.head()

Data has been extracted from the source system as '<class 'sklearn.utils._bunch.Bunch'>'
Raw, untouched data has been saved to: ../data/raw/iris_raw_v1.csv


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


In [16]:
# Verify data types and check for missing values
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   sepal length (cm)  150 non-null    float64
 1   sepal width (cm)   150 non-null    float64
 2   petal length (cm)  150 non-null    float64
 3   petal width (cm)   150 non-null    float64
 4   target             150 non-null    int64  
dtypes: float64(4), int64(1)
memory usage: 6.0 KB


### 4.2. Transform: Standardize the Raw Data
Now, we perform the **Transform** step. We load the raw data from `/data/raw/`, apply lightweight corrections like standardizing column names, and prepare it for loading into the `interim` layer.

In [12]:
# Load the raw data we just saved
interim_df = pd.read_csv(raw_data_path)

# Standardize column names to snake_case
new_columns = [col.replace(' (cm)', '').replace(' ', '_') for col in interim_df.columns]
interim_df.columns = new_columns

# Add the human-readable class name for context
interim_df['class_name'] = raw_data.target_names[interim_df['target']]

print("Columns have been transformed to snake_case.")
interim_df.head()

Columns have been transformed to snake_case.


,sepal_length,sepal_width,petal_length,petal_width,target,class_name
0,5.1,3.5,1.4,0.2,0,setosa
1,4.9,3.0,1.4,0.2,0,setosa
2,4.7,3.2,1.3,0.2,0,setosa
3,4.6,3.1,1.5,0.2,0,setosa
4,5.0,3.6,1.4,0.2,0,setosa


### 4.3. Load: Save the Interim Data
Now we save the cleaned, transformed DataFrame to the `/data/interim/` directory. This file now becomes the official, clean starting point for all analysis.

In [13]:
# Define and create the interim data directory
interim_data_dir = Path("../data/interim")
interim_data_dir.mkdir(parents=True, exist_ok=True)

# Define the file path and save the transformed DataFrame
interim_data_path = interim_data_dir / "iris_multi_class__interim_v1.parquet"
interim_df.to_parquet(interim_data_path, index=False)

### 4.4. Verification
Finally, let's perform a sanity check. We'll load the `interim` data we just created and verify its structure and content.

In [14]:
# Load the final interim data and verify its contents
iris_df = pd.read_parquet(interim_data_path)

print("Verification successful. The following DataFrame is ready for analysis:")
iris_df.sample(5)

Verification successful. The following DataFrame is ready for analysis:


,sepal_length,sepal_width,petal_length,petal_width,target,class_name
103,6.3,2.9,5.6,1.8,2,virginica
118,7.7,2.6,6.9,2.3,2,virginica
117,7.7,3.8,6.7,2.2,2,virginica
125,7.2,3.2,6.0,1.8,2,virginica
100,6.3,3.3,6.0,2.5,2,virginica



---

> ### 📝 4. Data Collection Report (Summary)
>
> The data collection stage was successfully executed, following a formal **Extract, Transform, Load (ETL)** process to ensure reproducibility and clear data lineage.
>
> *   **Data Extraction & Raw Storage:** The raw Iris dataset was programmatically extracted from the Scikit-learn library (`load_iris()`). To create a "source of truth," this untouched data, with its original column names (e.g., `sepal length (cm)`), was saved to `/data/raw/iris_raw_v1.csv`. An initial verification step confirmed that the raw dataset contains **150 rows and 6 columns** with **no missing values**. All data types are appropriate for analysis.
>
> *   **Data Transformation:** The raw data was then loaded, and lightweight transformations were applied. The primary transformation was standardizing all column names to a consistent `snake_case` format (e.g., `sepal_length`) for easier access in subsequent analysis. The human-readable `class_name` was also added for context.
>
> *   **Data Loading:** The cleaned, transformed DataFrame was loaded into the project's analytical starting point: `/data/interim/iris_multi_class_v1.parquet`. This file serves as the official, versioned dataset for this lab.
>
> *   **Initial Verification:** A final verification step confirmed that the interim dataset loads correctly.
>
> **Status:** The data collection stage is **complete**. The resulting `iris_multi_class_v1.parquet` file is clean, verified, and ready for the Data Understanding stage.

---